# GNN Bipartite V1LD — Overfit sur une instance (dataset train_base)

Parallèle à `gnn/experiment_gnn_v1LD_base.ipynb` mais avec l'architecture bipartite.

**Graphe** : `HeteroData` item ↔ scénario, `O(n·N)` arêtes (vs `O(n·N²)` en homogène).  
**Modèle** : `GNNBipartite` — 3 aller-retours `GeneralConv` avec edge features `4m`.  
**Loss** : `lagrangian_v1ld` — identique au pipeline homogène.  
**Référence** : subgradient T=200 cold start λ=0.

## 1. Setup

In [ ]:
import sys, importlib
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

REPO = Path("../").resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# Pour les petits graphes (n=10, E≤500), l'overhead de synchronisation
# multi-thread dépasse largement le calcul : 32 threads → ~560 ms/conv,
# 1 thread → 0.1 ms/conv. Forcer 1 thread pour l'expérience locale.
torch.set_num_threads(1)

import gnn_bipartite.train_gnn_bipartite as TB
import gnn.train_gnn_v1ld as T
importlib.reload(TB)
importlib.reload(T)

from gnn_bipartite.train_gnn_bipartite import (
    build_graph_bipartite,
    LazyBipartiteDataset,
    GNNBipartite,
)
from gnn.train_gnn_v1ld import (
    load_raw_nodes,
    lagrangian_v1ld,
    verify_v1ld,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

print(f"torch threads  : {torch.get_num_threads()}")
print(f"[bnb_solver] C++ disponible : {T._USE_CPP_BNB}")

## 2. Chargement des données (train_base)

In [2]:
N_VAL = 10   # taille des instances
M     = 5    # nombre de contraintes (charge instances_n{N_VAL}_m{M}.txt)

DATA_DIR = REPO / "data" / "train_base"
print(f"DATA_DIR = {DATA_DIR}")

inst_file  = DATA_DIR / f"instances_n{N_VAL}_m{M}.txt"
nodes_file = DATA_DIR / f"nodes_n{N_VAL}_m{M}.txt"

instances, raw_records = load_raw_nodes(inst_file, nodes_file)
dataset = LazyBipartiteDataset(instances, raw_records)
print(f"\n{len(dataset)} enregistrements chargés  ({len(instances)} instances)")

# Affichage des 20 premiers graphes (construction lazy → build_graph_bipartite)
print(f"\n{'idx':>4}  {'depth':>5}  {'n_g':>4}  {'N_z':>4}  {'N':>4}  {'m':>3}  "
      f"{'E=n_g*N_z':>10}  {'item.x':>8}  {'scen.x':>8}  {'opt':>8}")
print("-" * 88)
for i in range(min(20, len(dataset))):
    g = dataset[i]
    E = g.n_g * g.N_z
    print(f"{i:4d}  {int(g.context[0].item()):5d}  {g.n_g:4d}  {g.N_z:4d}  {g.N:4d}  {g.m:3d}  "
          f"{E:10d}  {str(tuple(g['item'].x.shape)):>8}  "
          f"{str(tuple(g['scen'].x.shape)):>8}  {g.opt.item():8.2f}")
if len(dataset) > 20:
    print(f"  ... ({len(dataset)} graphes au total)")

DATA_DIR = /project/6062881/romanl/Learning_bounds_CCCP/data/train_base
[load_raw_nodes] 163349 enregistrements chargés, 4250 ignorés

163349 enregistrements chargés  (862 instances)

 idx  depth   n_g   N_z     N    m   E=n_g*N_z    item.x    scen.x       opt
----------------------------------------------------------------------------------------
   0      0    10    50    50    5         500   (10, 2)  (50, 11)   2085.41
   1      1     9    50    50    5         450    (9, 2)  (50, 11)   2085.41
   2      2     8    50    50    5         400    (8, 2)  (50, 11)   2085.41
   3      3     7    50    50    5         350    (7, 2)  (50, 11)   2085.41
   4      4     5    50    50    5         250    (5, 2)  (50, 11)   2085.41
   5      5     4    50    50    5         200    (4, 2)  (50, 11)   2085.41
   6      6     3    50    50    5         150    (3, 2)  (50, 11)   2085.41
   7      7     2    50    50    5         100    (2, 2)  (50, 11)   2085.41
   8      4     6    50    50    5

## 3. Références : L1(λ=0) et subgradient

In [3]:
GRAPH_IDX = 0
g = dataset[GRAPH_IDX]

with torch.no_grad():
    lam0    = torch.zeros(g.N_z * g.n_g)
    l1_init = lagrangian_v1ld(lam0, g).item()

opt_val      = g.opt.item()
curr_cost    = g.context[3].item()   # index 3 = curr_cost
opt_residual = opt_val - curr_cost
depth        = int(g.context[0].item())

print(f"Graphe {GRAPH_IDX}  (depth={depth}, n_g={g.n_g}, N_z={g.N_z}/{g.N}, m={g.m})")
print(f"  ε              = {g.epsilon.item():.2f}   n_fixed_z1={g.n_fixed_z1}")
print(f"  curr_cost      = {curr_cost:.2f}")
print(f"  opt (global)   = {opt_val:.2f}")
print(f"  opt_résiduel   = {opt_residual:.2f}")
print(f"  L1(λ=0)        = {l1_init:.2f}   ← borne de départ")
print()
print(f"Graphe bipartite :")
print(f"  item.x         = {tuple(g['item'].x.shape)}")
print(f"  scen.x         = {tuple(g['scen'].x.shape)}")
print(f"  edge_attr      = {tuple(g['item','to','scen'].edge_attr.shape)}  (E = n_g × N_z)")
print()

PRECISION = 1
PATIENCE  = 30
print(f"Subgradient convergence (T=500, precision={PRECISION}, patience={PATIENCE})...")
# verify_v1ld accède uniquement aux attributs scalaires/tenseurs → compatible HeteroData
ref_subgrad, n_iters_ref = verify_v1ld(
    g, T=500, precision=PRECISION, patience=PATIENCE, return_n_iters=True)
print(f"  convergé : {ref_subgrad:.2f}  ({n_iters_ref} it.)  "
      f"({(l1_init - ref_subgrad)/l1_init*100:.1f}% vs L1(0))")

Graphe 0  (depth=0, n_g=10, N_z=50/50, m=5)
  ε              = 0.10   n_fixed_z1=0
  curr_cost      = 0.00
  opt (global)   = 2085.41
  opt_résiduel   = 2085.41
  L1(λ=0)        = 3307.53   ← borne de départ

Graphe bipartite :
  item.x         = (10, 2)
  scen.x         = (50, 11)
  edge_attr      = (500, 20)  (E = n_g × N_z)

Subgradient convergence (T=500, precision=1, patience=30)...
  convergé : 2940.62  (500 it.)  (11.1% vs L1(0))


## 4. Configuration de l'entraînement

In [4]:
CFG = dict(
    n_epochs  = 3000,
    lr        = 1e-3,
    scheduler = "cosine",
    eta_min   = 1e-4,
    log_every = 50,
)

## 5. Entraînement (overfit sur un graphe)

In [ ]:
torch.set_num_threads(1)   # critique : 32 threads → 693× plus lent sur petits graphes
torch.manual_seed(42)
device = torch.device("cpu")
data   = g   # HeteroData — déjà construit

model     = GNNBipartite(m=data.m, hidden=64).to(device)
n_params  = sum(p.numel() for p in model.parameters())
print(f"GNNBipartite — m={data.m}  hidden=64  params={n_params:,}")
print(f"torch threads  : {torch.get_num_threads()}")

optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])

if CFG["scheduler"] == "cosine":
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG["n_epochs"], eta_min=CFG["eta_min"])
else:
    sched = None

history = []
import time as _time
t_start = _time.time()

for epoch in range(1, CFG["n_epochs"] + 1):
    model.train()
    optimizer.zero_grad()
    lam  = model(data)
    loss = lagrangian_v1ld(lam, data)
    loss.backward()
    optimizer.step()
    if sched: sched.step()

    current_lr = optimizer.param_groups[0]["lr"]
    history.append((epoch, loss.item(), current_lr))

    if epoch % CFG["log_every"] == 0 or epoch == 1:
        b = loss.item()
        gap_res = (b - opt_residual) / max(abs(opt_residual), 1e-9) * 100
        gap_l0  = (b - l1_init) / l1_init * 100
        elapsed = _time.time() - t_start
        ms_ep   = elapsed / epoch * 1000
        print(f"epoch {epoch:5d}  L1={b:9.2f}  "
              f"gap_res={gap_res:+6.1f}%  "
              f"gap_L1_0={gap_l0:+6.1f}%  "
              f"lr={current_lr:.2e}  ({ms_ep:.0f}ms/ep)")

final_l1 = history[-1][1]
total_min = (_time.time() - t_start) / 60
print(f"\nBorne résiduelle finale : {final_l1:.2f}  "
      f"(+ curr_cost {curr_cost:.2f} = {final_l1+curr_cost:.2f} vs opt {opt_val:.2f})")
print(f"  vs L1(λ=0)        : {l1_init:.2f} ({(l1_init - final_l1)/l1_init*100:.1f}% amélioration)")
print(f"  vs subgradient ref: {ref_subgrad:.2f} ({'mieux' if final_l1 < ref_subgrad else 'moins bon'})")
print(f"  Temps total        : {total_min:.1f} min")

## 6. Résultats : GNN seul vs warm start vs subgradient

In [ ]:
model.eval()
with torch.no_grad():
    lam_gnn = model(data).numpy().reshape(g.N_z, g.n_g).astype(np.float64)

l1_warm_5  = verify_v1ld(g, T=5,  lam_init=lam_gnn)
l1_warm_20 = verify_v1ld(g, T=20, lam_init=lam_gnn)
l1_cold_60 = verify_v1ld(g, T=60)
l1_conv, n_iters_conv = verify_v1ld(
    g, T=500, precision=PRECISION, patience=PATIENCE, return_n_iters=True)

rows = [
    ("L1(λ=0)  [départ]",                                      l1_init),
    ("GNN bipartite seul",                                      final_l1),
    ("GNN bipartite + 5 it. subgrad  [warm start]",            l1_warm_5),
    ("GNN bipartite + 20 it. subgrad [warm start]",            l1_warm_20),
    ("Subgradient 60 it.  [cold start]",                       l1_cold_60),
    (f"Subgradient convergé  [{n_iters_conv} it., prec={PRECISION}, pat={PATIENCE}]",
                                                                l1_conv),
    (f"opt_résiduel = opt−curr_cost  [{opt_val:.0f}−{curr_cost:.0f}]",
                                                                opt_residual),
]

W = max(len(r[0]) for r in rows)
print(f"{'':>{W}}   {'L1_res':>10}   {'vs opt_res':>10}   {'vs L1(0)':>10}")
print("─" * (W + 40))
for label, val in rows:
    gap_res = (val - opt_residual) / max(abs(opt_residual), 1e-9) * 100
    gap_l0  = (val - l1_init) / l1_init * 100
    marker  = "  ←" if "GNN" in label else ""
    print(f"{label:>{W}}   {val:10.2f}   {gap_res:+9.1f}%   {gap_l0:+9.1f}%{marker}")

print()
print(f"Gain warm start  5 it. vs GNN seul : {final_l1 - l1_warm_5:+.2f}")
print(f"Gain warm start 20 it. vs GNN seul : {final_l1 - l1_warm_20:+.2f}")

## 7. Visualisation

In [ ]:
epochs_h = [h[0] for h in history]
l1_h     = [h[1] for h in history]
lr_h     = [h[2] for h in history]

W = 20
l1_smooth = np.convolve(l1_h, np.ones(W)/W, mode="valid")
ep_smooth = epochs_h[W-1:]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7),
                                gridspec_kw={"height_ratios": [3, 1]},
                                sharex=True)

ax1.plot(epochs_h, l1_h,    alpha=0.25, color="steelblue", lw=0.8, label="L1(λ_GNN) brut")
ax1.plot(ep_smooth, l1_smooth, color="steelblue", lw=2,
         label=f"L1(λ_GNN) lissé (W={W})")

ax1.axhline(l1_init,      color="orange", lw=1.5, ls="--",
            label=f"L1(λ=0) = {l1_init:.0f}")
ax1.axhline(ref_subgrad,  color="red",    lw=1.5, ls="--",
            label=f"Subgradient (prec={PRECISION}, pat={PATIENCE}) = {ref_subgrad:.0f}")
ax1.axhline(opt_residual, color="green",  lw=1.5, ls=":",
            label=f"opt_résiduel = {opt_residual:.0f}")

ax1.scatter([epochs_h[-1]], [final_l1], color="steelblue", s=60, zorder=5)
ax1.annotate(f"  {final_l1:.0f}", (epochs_h[-1], final_l1),
             va="center", fontsize=9, color="steelblue")

ax1.set_ylabel("L1(λ) résiduelle")
ax1.set_title(
    f"Convergence GNN Bipartite V1LD (train_base) — "
    f"depth={depth}, n_g={g.n_g}, N_z={g.N_z}/{g.N}, m={g.m}\n"
    f"scheduler={CFG['scheduler']}, lr={CFG['lr']}, epochs={CFG['n_epochs']}"
)
ax1.legend(loc="upper right", fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))
ax1.grid(True, alpha=0.3)

ax2.semilogy(epochs_h, lr_h, color="gray", lw=1.5)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("LR")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
out_fig = REPO / "gnn_bipartite" / "convergence_bipartite_base.png"
plt.savefig(out_fig, bbox_inches="tight")
print(f"Figure sauvegardée : {out_fig}")
plt.show()

## 8. (Optionnel) Comparaison architectures : bipartite vs homogène

Compare les deux modèles sur le même graphe et la même loss.

In [ ]:
from gnn.train_gnn_v1ld import build_graph, GNN

# Reconstruire le graphe homogène à partir du même node_rec
rec  = raw_records[GRAPH_IDX]
inst = instances[rec["inst_id"]]

b_res = rec["b_res_explicit"] if rec["b_res_explicit"] is not None else inst["b"].copy()
node_rec_hom = {
    "depth":     rec["depth"],
    "curr_cost": rec["curr_cost"],
    "x_mask":    rec["x_mask"],
    "b_res":     b_res,
}
if rec["z_mask"] is not None:
    node_rec_hom["z_mask"] = rec["z_mask"]

g_hom = build_graph(inst, node_rec_hom)
print(f"Graphe homogène : {g_hom.x.shape[0]} nœuds, {g_hom.edge_index.shape[1]} arêtes")
print(f"Graphe bipartite: {g.n_g} items + {g.N_z} scénarios = {g.n_g+g.N_z} nœuds, {g.n_g*g.N_z} arêtes (× 2 dir.)")


def run_overfit(model, data, n_epochs, lr, eta_min, label, seed=42):
    torch.manual_seed(seed)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=n_epochs, eta_min=eta_min)
    hist  = []
    for epoch in range(1, n_epochs + 1):
        model.train(); opt.zero_grad()
        lam  = model(data)
        loss = lagrangian_v1ld(lam, data)
        loss.backward(); opt.step(); sched.step()
        hist.append(loss.item())
        if epoch % 200 == 0 or epoch == 1:
            print(f"  [{label}] epoch {epoch:5d}  L1={loss.item():9.2f}")
    return hist


N_COMP = 1000
print(f"\n--- GNN Bipartite ({sum(p.numel() for p in GNNBipartite(m=g.m).parameters()):,} params) ---")
m_bip = GNNBipartite(m=g.m, hidden=64)
hist_bip = run_overfit(m_bip, data, N_COMP, lr=1e-3, eta_min=1e-4, label="bipartite")

print(f"\n--- GNN Homogène ({sum(p.numel() for p in GNN(m=g.m).parameters()):,} params) ---")
m_hom = GNN(m=g.m, hidden=64)
hist_hom = run_overfit(m_hom, g_hom, N_COMP, lr=1e-3, eta_min=1e-4, label="homogène")

print(f"\nFinal bipartite : {hist_bip[-1]:.2f}")
print(f"Final homogène  : {hist_hom[-1]:.2f}")
print(f"Subgradient ref : {ref_subgrad:.2f}")

In [ ]:
W = 20
ep = list(range(1, N_COMP + 1))

sm_bip = np.convolve(hist_bip, np.ones(W)/W, mode="valid")
sm_hom = np.convolve(hist_hom, np.ones(W)/W, mode="valid")
ep_sm  = ep[W-1:]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ep[W-1:], sm_bip, color="steelblue", lw=2, label=f"GNN bipartite (final={hist_bip[-1]:.0f})")
ax.plot(ep[W-1:], sm_hom, color="darkorange", lw=2, label=f"GNN homogène  (final={hist_hom[-1]:.0f})")

ax.axhline(l1_init,      color="gray",  lw=1.5, ls="--", label=f"L1(λ=0)={l1_init:.0f}")
ax.axhline(ref_subgrad,  color="red",   lw=1.5, ls="--", label=f"Subgrad ref={ref_subgrad:.0f}")
ax.axhline(opt_residual, color="green", lw=1.5, ls=":",  label=f"opt_res={opt_residual:.0f}")

ax.set_xlabel("Epoch"); ax.set_ylabel("L1(λ) résiduelle lissée")
ax.set_title(f"Bipartite vs Homogène — overfit graphe {GRAPH_IDX} "
             f"(n_g={g.n_g}, N_z={g.N_z}, m={g.m}, {N_COMP} epochs)")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))

plt.tight_layout()
out_fig2 = REPO / "gnn_bipartite" / "convergence_bipartite_vs_homogene.png"
plt.savefig(out_fig2, bbox_inches="tight")
print(f"Figure sauvegardée : {out_fig2}")
plt.show()

## 9. (Optionnel) Sweep de configs

In [ ]:
def run_config_bip(cfg, data, seed=42):
    torch.manual_seed(seed)
    model_c = GNNBipartite(m=data.m, hidden=cfg.get("hidden", 64))
    opt     = torch.optim.Adam(model_c.parameters(), lr=cfg["lr"])
    n_ep    = cfg["n_epochs"]
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=n_ep, eta_min=cfg.get("eta_min", 1e-5))
    hist = []
    for epoch in range(1, n_ep + 1):
        model_c.train(); opt.zero_grad()
        lam  = model_c(data)
        loss = lagrangian_v1ld(lam, data)
        loss.backward(); opt.step(); sched.step()
        hist.append((epoch, loss.item()))
    print(f"  {cfg['label']:25s}  final={hist[-1][1]:.2f}")
    return hist


CONFIGS = [
    dict(label="cosine lr=1e-3 h=64",  n_epochs=1000, lr=1e-3, eta_min=1e-4, hidden=64),
    dict(label="cosine lr=3e-3 h=64",  n_epochs=1000, lr=3e-3, eta_min=1e-4, hidden=64),
    dict(label="cosine lr=1e-3 h=128", n_epochs=1000, lr=1e-3, eta_min=1e-4, hidden=128),
    dict(label="cosine lr=1e-4 h=64",  n_epochs=1000, lr=1e-4, eta_min=1e-5, hidden=64),
]

results = {}
for cfg in CONFIGS:
    results[cfg["label"]] = run_config_bip(cfg, data)

fig, ax = plt.subplots(figsize=(10, 5))
colors  = plt.cm.tab10.colors
W = 30
for (label, hist), color in zip(results.items(), colors):
    ep   = [h[0] for h in hist]
    vals = [h[1] for h in hist]
    sm   = np.convolve(vals, np.ones(W)/W, mode="valid")
    ax.plot(ep[W-1:], sm, color=color, lw=2, label=label)

ax.axhline(l1_init,      color="gray",  lw=1.5, ls="--", label=f"L1(λ=0)={l1_init:.0f}")
ax.axhline(ref_subgrad,  color="red",   lw=1.5, ls="--", label=f"Subgrad={ref_subgrad:.0f}")
ax.axhline(opt_residual, color="green", lw=1.5, ls=":",  label=f"opt_res={opt_residual:.0f}")

ax.set_xlabel("Epoch"); ax.set_ylabel("L1(λ) résiduelle lissée")
ax.set_title("Sweep configs — GNN Bipartite V1LD train_base")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))
plt.tight_layout()
plt.savefig(REPO / "gnn_bipartite" / "convergence_bipartite_sweep.png", bbox_inches="tight")
plt.show()